# Лабораторная работа №2. Проведение экспериментов по настройке модели

**Датасет:** Car Price Prediction — Used Cars  
**Тип задачи:** регрессия  
**Целевая переменная:** `Selling_Price`

В работе последовательно выполняются baseline-модель, feature engineering, feature selection, подбор гиперпараметров Optuna, дополнительный RFE и CatBoost, логирование в MLflow и подготовка Production-модели.


## 0. Подготовка

Перед выполнением notebook MLflow должен быть запущен в отдельном терминале:

```bash
cd mlflow
sh start_mlflow.sh
```

Интерфейс: `http://127.0.0.1:5000/`.


In [1]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, PolynomialFeatures, KBinsDiscretizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
from sklearn.feature_selection import RFE

from mlxtend.feature_selection import SequentialFeatureSelector as SFS
import optuna

import mlflow
import mlflow.sklearn
import mlflow.catboost
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

from catboost import CatBoostRegressor

RANDOM_STATE = 42
TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENT_NAME = "LR2_Car_Price_Prediction"
MODEL_NAME = "car_price_prediction_model"


/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Определяем корень проекта независимо от того, запущен notebook из research/ или из корня.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [cwd, cwd.parent] if (p / "data" / "clean_data.pkl").exists()),
    cwd.parent
)
DATA_PATH = PROJECT_ROOT / "data" / "clean_data.pkl"
RESEARCH_DIR = PROJECT_ROOT / "research"
REQUIREMENTS_PATH = PROJECT_ROOT / "requirements.txt"
README_PATH = PROJECT_ROOT / "README.md"

RESEARCH_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
assert DATA_PATH.exists(), "Не найден data/clean_data.pkl. Сначала выполните ЛР1."


PROJECT_ROOT: /home/webber/iis/lab1_car_price
DATA_PATH: /home/webber/iis/lab1_car_price/data/clean_data.pkl


## 1. Загрузка очищенной выборки и train/test split 75%/25%


In [3]:
df = pd.read_pickle(DATA_PATH)
TARGET = "Selling_Price"

X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(exclude=np.number).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

print("Размер очищенной выборки:", df.shape)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Числовые признаки:", numeric_cols)
print("Категориальные признаки:", categorical_cols)
display(df.head())


Размер очищенной выборки: (286, 10)
Train: (214, 9) Test: (72, 9)
Числовые признаки: ['Year', 'Present_Price', 'Driven_kms', 'Owner', 'Car_Age']
Категориальные признаки: ['Car_Name', 'Fuel_Type', 'Selling_type', 'Transmission']


,Car_Name,Year,Selling_Price,Present_Price,Driven_kms,Fuel_Type,Selling_type,Transmission,Owner,Car_Age
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0,4
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0,5
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0,1
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0,7
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0,4


## 2. Служебные функции и подключение MLflow


In [4]:
def regression_metrics(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "mape": float(mean_absolute_percentage_error(y_true, y_pred)),
        "mse": float(mean_squared_error(y_true, y_pred)),
    }


def print_metrics(name, metrics):
    print(name)
    for key, value in metrics.items():
        print(f"  {key}: {value:.6f}")


def save_lines(path, values):
    Path(path).write_text("\n".join(map(str, values)), encoding="utf-8")


class IndexSelector(BaseEstimator, TransformerMixin):
    """Выбирает столбцы numpy-массива по фиксированным индексам."""
    def __init__(self, indices):
        self.indices = indices

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[:, list(self.indices)]


mlflow.set_tracking_uri(TRACKING_URI)
experiment = mlflow.set_experiment(EXPERIMENT_NAME)
EXPERIMENT_ID = experiment.experiment_id
client = MlflowClient()

(RESEARCH_DIR / "experiment_id.txt").write_text(str(EXPERIMENT_ID), encoding="utf-8")
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment ID:", EXPERIMENT_ID)


2026/09/22 23:49:57 INFO mlflow.tracking.fluent: Experiment with name 'LR2_Car_Price_Prediction' does not exist. Creating a new experiment.


MLflow tracking URI: http://127.0.0.1:5000
Experiment ID: 1


In [5]:
def latest_model_version(model_name=MODEL_NAME):
    versions = client.search_model_versions(f"name='{model_name}'")
    if not versions:
        return None
    return max(int(v.version) for v in versions)


def log_sklearn_run(run_name, fitted_model, metrics, X_example, params=None,
                    artifact_paths=None, register=False, tags=None):
    signature = infer_signature(
        X_example,
        fitted_model.predict(X_example)
    )
    with mlflow.start_run(run_name=run_name) as run:
        if params:
            mlflow.log_params(params)
        if metrics:
            mlflow.log_metrics(metrics)
        if tags:
            mlflow.set_tags(tags)
        if REQUIREMENTS_PATH.exists():
            mlflow.log_artifact(str(REQUIREMENTS_PATH), artifact_path="project")
        for path in artifact_paths or []:
            if Path(path).exists():
                mlflow.log_artifact(str(path), artifact_path="research")

        mlflow.sklearn.log_model(
            sk_model=fitted_model,
            artifact_path="model",
            signature=signature,
            input_example=X_example,
            registered_model_name=MODEL_NAME if register else None,
        )
        run_id = run.info.run_id
    return run_id


experiment_results = {}
run_ids = {}


## 3. Baseline: StandardScaler + OrdinalEncoder + RandomForest


In [6]:
baseline_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_cols),
    ],
    remainder="drop"
)

baseline_model = Pipeline([
    ("transform", baseline_transformer),
    ("regression", RandomForestRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )),
])

baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)
baseline_metrics = regression_metrics(y_test, baseline_pred)
experiment_results["baseline_random_forest"] = baseline_metrics
print_metrics("Baseline", baseline_metrics)


Baseline
  mae: 0.636031
  mape: 0.138466
  mse: 2.285179


In [7]:
run_ids["baseline_random_forest"] = log_sklearn_run(
    run_name="01_baseline_random_forest",
    fitted_model=baseline_model,
    metrics=baseline_metrics,
    X_example=X_train.head(5),
    params={"n_estimators": 100, "random_state": RANDOM_STATE},
    register=True,
    tags={"stage": "baseline", "lab": "LR2"},
)
print("Baseline run_id:", run_ids["baseline_random_forest"])
print("Зарегистрирована версия модели:", latest_model_version())


/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Successfully registered model 'car_price_prediction_model'.
2026/09/22 23:49:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: car_price_prediction_model, version 1


Created version '1' of model 'car_price_prediction_model'.


2026/09/22 23:49:59 INFO mlflow.tracking._tracking_service.client: 🏃 View run 01_baseline_random_forest at: http://127.0.0.1:5000/#/experiments/1/runs/d6c73cbe995642a6ad6ce2980865fa70.


2026/09/22 23:49:59 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


Baseline run_id: d6c73cbe995642a6ad6ce2980865fa70
Зарегистрирована версия модели: 1


## 4. Feature engineering средствами sklearn

Для трех числовых признаков используются `PolynomialFeatures(degree=2)`. Так как выборка не является временным рядом, для тех же трех числовых признаков используется `KBinsDiscretizer`. Базовые StandardScaler/OrdinalEncoder сохраняются.


In [8]:
poly_cols = ["Present_Price", "Driven_kms", "Car_Age"]

poly_pipeline = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scale", StandardScaler()),
])

bins_pipeline = Pipeline([
    ("bins", KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile")),
    ("scale", StandardScaler()),
])

feature_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_cols),
        ("poly", poly_pipeline, poly_cols),
        ("bins", bins_pipeline, poly_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

# По методичке сначала создаем копию исходной train-выборки.
X_train_fe_sklearn = X_train.copy()

X_train_fe_array = feature_transformer.fit_transform(X_train_fe_sklearn, y_train)
X_test_fe_array = feature_transformer.transform(X_test)
feature_names = feature_transformer.get_feature_names_out().tolist()

X_train_fe_sklearn = pd.DataFrame(
    X_train_fe_array, columns=feature_names, index=X_train.index
)
X_test_fe_sklearn = pd.DataFrame(
    X_test_fe_array, columns=feature_names, index=X_test.index
)

FEATURE_NAMES_PATH = RESEARCH_DIR / "feature_names_sklearn.txt"
save_lines(FEATURE_NAMES_PATH, feature_names)

print("Количество признаков после feature engineering:", len(feature_names))
display(X_train_fe_sklearn.head())


Количество признаков после feature engineering: 21


,num__Year,num__Present_Price,num__Driven_kms,num__Owner,num__Car_Age,cat__Car_Name,cat__Fuel_Type,cat__Selling_type,cat__Transmission,poly__Present_Price,...,poly__Car_Age,poly__Present_Price^2,poly__Present_Price Driven_kms,poly__Present_Price Car_Age,poly__Driven_kms^2,poly__Driven_kms Car_Age,poly__Car_Age^2,bins__Present_Price,bins__Driven_kms,bins__Car_Age
16,0.468974,0.356067,0.189734,0.0,-0.468974,61.0,0.0,0.0,1.0,0.356067,...,-0.468974,-0.038840,0.129571,-0.042551,-0.093841,-0.277086,-0.490710,1.318621,0.428013,0.135229
161,0.125274,-0.735010,-0.445571,0.0,-0.125274,43.0,1.0,1.0,1.0,-0.735010,...,-0.125274,-0.227723,-0.478914,-0.503915,-0.357173,-0.383560,-0.305176,-1.335155,-0.469975,0.135229
273,-1.249526,-0.004194,0.853849,0.0,1.249526,70.0,1.0,0.0,1.0,-0.004194,...,1.249526,-0.137040,0.142733,0.396020,0.344027,0.898055,0.967052,0.434029,1.326000,1.168768
71,-0.905826,0.541125,0.302728,0.0,0.905826,56.0,0.0,0.0,1.0,0.541125,...,0.905826,0.025337,0.287650,0.830305,-0.031075,0.335210,0.569481,1.318621,0.428013,1.168768
120,0.812673,-0.687486,-1.034005,0.0,-0.812673,15.0,1.0,1.0,1.0,-0.687486,...,-0.812673,-0.226245,-0.495608,-0.516359,-0.465402,-0.643349,-0.623233,-0.450563,-1.367962,-0.898310


In [9]:
feature_engineering_model = Pipeline([
    ("transform", feature_transformer),
    ("regression", RandomForestRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )),
])

feature_engineering_model.fit(X_train, y_train)
fe_pred = feature_engineering_model.predict(X_test)
fe_metrics = regression_metrics(y_test, fe_pred)
experiment_results["sklearn_feature_engineering"] = fe_metrics
print_metrics("Sklearn feature engineering", fe_metrics)

run_ids["sklearn_feature_engineering"] = log_sklearn_run(
    run_name="02_sklearn_feature_engineering",
    fitted_model=feature_engineering_model,
    metrics=fe_metrics,
    X_example=X_train.head(5),
    params={"degree": 2, "n_bins": 4, "n_estimators": 100},
    artifact_paths=[FEATURE_NAMES_PATH],
    tags={"stage": "feature_engineering", "lab": "LR2"},
)


/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Sklearn feature engineering
  mae: 0.659286
  mape: 0.135843
  mse: 2.522464



2026/09/22 23:50:00 INFO mlflow.tracking._tracking_service.client: 🏃 View run 02_sklearn_feature_engineering at: http://127.0.0.1:5000/#/experiments/1/runs/2e9f28fed6e4444fa4b00373e71adc37.


2026/09/22 23:50:00 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


## 5. Feature selection с mlxtend: SequentialFeatureSelector, forward=True

После feature engineering получается около двух десятков признаков. Выбираем 8 признаков, то есть примерно треть от расширенного набора, что находится внутри рекомендованного диапазона 20–70%.


In [10]:
N_SELECTED = min(8, max(1, len(feature_names)))

sfs = SFS(
    estimator=RandomForestRegressor(
        n_estimators=20,
        max_depth=8,
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
    k_features=N_SELECTED,
    forward=True,
    floating=False,
    scoring="neg_mean_absolute_error",
    cv=2,
    n_jobs=1,
    verbose=0,
)

sfs.fit(X_train_fe_sklearn.values, y_train.values)
selected_indices = list(map(int, sfs.k_feature_idx_))
selected_names = [feature_names[i] for i in selected_indices]

SELECTED_NAMES_PATH = RESEARCH_DIR / "selected_feature_names.txt"
SELECTED_INDICES_PATH = RESEARCH_DIR / "selected_feature_indices.txt"
save_lines(SELECTED_NAMES_PATH, selected_names)
save_lines(SELECTED_INDICES_PATH, selected_indices)

print("Индексы выбранных признаков:", selected_indices)
print("Названия выбранных признаков:")
for name in selected_names:
    print(" -", name)


/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(
/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(
/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(
/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(
/home/we

/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(


/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(


/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(
/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(


/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(


/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(
/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(
/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(
/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(
/home/we

... (33 outputs left out)

/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(
/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(


/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(
/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(
/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(


Индексы выбранных признаков: [0, 1, 3, 4, 5, 6, 8, 19]
Названия выбранных признаков:
 - num__Year
 - num__Present_Price
 - num__Owner
 - num__Car_Age
 - cat__Car_Name
 - cat__Fuel_Type
 - cat__Transmission
 - bins__Driven_kms


/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:73: FutureWarning: `fit_params` is deprecated and will be removed in version 1.6. Pass parameters via `params` instead.
  warnings.warn(


In [11]:
sfs_model = Pipeline([
    ("transform", feature_transformer),
    ("feature_selection", IndexSelector(selected_indices)),
    ("regression", RandomForestRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

sfs_model.fit(X_train, y_train)
sfs_pred = sfs_model.predict(X_test)
sfs_metrics = regression_metrics(y_test, sfs_pred)
experiment_results["forward_sfs"] = sfs_metrics
print_metrics("Forward SFS", sfs_metrics)

run_ids["forward_sfs"] = log_sklearn_run(
    run_name="03_forward_sfs_mlxtend",
    fitted_model=sfs_model,
    metrics=sfs_metrics,
    X_example=X_train.head(5),
    params={"n_selected": len(selected_indices), "direction": "forward"},
    artifact_paths=[FEATURE_NAMES_PATH, SELECTED_NAMES_PATH, SELECTED_INDICES_PATH],
    tags={"stage": "feature_selection", "selector": "mlxtend_sfs", "lab": "LR2"},
)


/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Forward SFS
  mae: 0.664955
  mape: 0.131208
  mse: 2.369676



2026/09/22 23:50:04 INFO mlflow.tracking._tracking_service.client: 🏃 View run 03_forward_sfs_mlxtend at: http://127.0.0.1:5000/#/experiments/1/runs/3cae138aa1924edb8e1c49be7b28b6b3.


2026/09/22 23:50:04 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


## 6. Дополнительное задание (*): RFE и сравнение выбранных признаков


In [12]:
rfe = RFE(
    estimator=RandomForestRegressor(
        n_estimators=50,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    n_features_to_select=N_SELECTED,
    step=2,
)
rfe.fit(X_train_fe_sklearn.values, y_train.values)

rfe_indices = np.flatnonzero(rfe.support_).astype(int).tolist()
rfe_names = [feature_names[i] for i in rfe_indices]

RFE_NAMES_PATH = RESEARCH_DIR / "rfe_feature_names.txt"
RFE_INDICES_PATH = RESEARCH_DIR / "rfe_feature_indices.txt"
save_lines(RFE_NAMES_PATH, rfe_names)
save_lines(RFE_INDICES_PATH, rfe_indices)

intersection = sorted(set(selected_names) & set(rfe_names))
union = set(selected_names) | set(rfe_names)
jaccard = len(intersection) / len(union) if union else 0.0

print("Совпадающие признаки SFS и RFE:", intersection)
print(f"Jaccard similarity: {jaccard:.3f}")


Совпадающие признаки SFS и RFE: ['cat__Car_Name', 'num__Present_Price', 'num__Year']
Jaccard similarity: 0.231


In [13]:
rfe_model = Pipeline([
    ("transform", feature_transformer),
    ("feature_selection", IndexSelector(rfe_indices)),
    ("regression", RandomForestRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])
rfe_model.fit(X_train, y_train)
rfe_pred = rfe_model.predict(X_test)
rfe_metrics = regression_metrics(y_test, rfe_pred)
experiment_results["optional_rfe"] = rfe_metrics
print_metrics("RFE", rfe_metrics)

run_ids["optional_rfe"] = log_sklearn_run(
    run_name="04_optional_rfe",
    fitted_model=rfe_model,
    metrics=rfe_metrics,
    X_example=X_train.head(5),
    params={"n_selected": len(rfe_indices), "selector": "RFE"},
    artifact_paths=[RFE_NAMES_PATH, RFE_INDICES_PATH],
    tags={"stage": "feature_selection", "selector": "RFE", "optional": "true", "lab": "LR2"},
)


RFE
  mae: 0.692535
  mape: 0.137637
  mse: 2.393862


/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(



2026/09/22 23:50:06 INFO mlflow.tracking._tracking_service.client: 🏃 View run 04_optional_rfe at: http://127.0.0.1:5000/#/experiments/1/runs/c702b8167cc14977aa6f89ff2342a004.


2026/09/22 23:50:06 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


## 7. Optuna: подбор гиперпараметров RandomForest

Среди трех обязательных вариантов RandomForest выбирается вариант с минимальным MAE. Затем выполняется не менее 10 trials. Для регрессии MAE необходимо **минимизировать**.


In [14]:
mandatory_rf_results = {
    "baseline_random_forest": baseline_metrics,
    "sklearn_feature_engineering": fe_metrics,
    "forward_sfs": sfs_metrics,
}
best_rf_variant = min(mandatory_rf_results, key=lambda name: mandatory_rf_results[name]["mae"])
print("Лучший вариант RandomForest до Optuna:", best_rf_variant)


def build_rf_variant(variant, rf_model):
    if variant == "baseline_random_forest":
        return Pipeline([
            ("transform", baseline_transformer),
            ("regression", rf_model),
        ])
    if variant == "sklearn_feature_engineering":
        return Pipeline([
            ("transform", feature_transformer),
            ("regression", rf_model),
        ])
    if variant == "forward_sfs":
        return Pipeline([
            ("transform", feature_transformer),
            ("feature_selection", IndexSelector(selected_indices)),
            ("regression", rf_model),
        ])
    raise ValueError(variant)


cv = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 18),
        "max_features": trial.suggest_float("max_features", 0.1, 1.0),
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    }
    candidate = build_rf_variant(best_rf_variant, RandomForestRegressor(**params))
    scores = cross_val_score(
        candidate,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=1,
    )
    return float(-scores.mean())

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=10)

print("Best CV MAE:", study.best_value)
print("Best params:", study.best_params)


Лучший вариант RandomForest до Optuna: baseline_random_forest


Best CV MAE: 0.7796130770735519
Best params: {'n_estimators': 200, 'max_depth': 18, 'max_features': 0.7587945476302645}


In [15]:
best_rf_params = {
    **study.best_params,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}
optuna_model = build_rf_variant(
    best_rf_variant,
    RandomForestRegressor(**best_rf_params)
)
optuna_model.fit(X_train, y_train)
optuna_pred = optuna_model.predict(X_test)
optuna_metrics = regression_metrics(y_test, optuna_pred)
experiment_results["optuna_random_forest"] = optuna_metrics
print_metrics("Optuna RandomForest", optuna_metrics)

run_ids["optuna_random_forest"] = log_sklearn_run(
    run_name="05_optuna_random_forest",
    fitted_model=optuna_model,
    metrics=optuna_metrics,
    X_example=X_train.head(5),
    params={**study.best_params, "source_variant": best_rf_variant, "trials": 10},
    register=True,
    tags={"stage": "hyperparameter_tuning", "optimizer": "optuna", "lab": "LR2"},
)
print("Optuna run_id:", run_ids["optuna_random_forest"])
print("Зарегистрирована версия модели:", latest_model_version())


Optuna RandomForest
  mae: 0.632630
  mape: 0.123129
  mse: 2.321851


/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Registered model 'car_price_prediction_model' already exists. Creating a new version of this model...
2026/09/22 23:50:12 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: car_price_prediction_model, version 2


Created version '2' of model 'car_price_prediction_model'.



2026/09/22 23:50:12 INFO mlflow.tracking._tracking_service.client: 🏃 View run 05_optuna_random_forest at: http://127.0.0.1:5000/#/experiments/1/runs/483ba6a69e7540369e39d97f09bde8a1.


2026/09/22 23:50:12 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


Optuna run_id: 483ba6a69e7540369e39d97f09bde8a1
Зарегистрирована версия модели: 2


## 8. Дополнительное задание (*): CatBoost

CatBoost обучается на исходных признаках. Категориальные признаки передаются в модель нативно. Модель логируется через `mlflow.catboost` flavour и регистрируется как очередная версия.


In [16]:
X_train_cat = X_train.copy()
X_test_cat = X_test.copy()
for col in categorical_cols:
    X_train_cat[col] = X_train_cat[col].astype(str)
    X_test_cat[col] = X_test_cat[col].astype(str)

catboost_model = CatBoostRegressor(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="MAE",
    random_seed=RANDOM_STATE,
    verbose=False,
)
catboost_model.fit(
    X_train_cat,
    y_train,
    cat_features=categorical_cols,
)
cat_pred = catboost_model.predict(X_test_cat)
cat_metrics = regression_metrics(y_test, cat_pred)
experiment_results["optional_catboost"] = cat_metrics
print_metrics("CatBoost", cat_metrics)


CatBoost
  mae: 0.779166
  mape: 0.263721
  mse: 2.286548


In [17]:
cat_signature = infer_signature(
    X_train_cat.head(5),
    catboost_model.predict(X_train_cat.head(5))
)
with mlflow.start_run(run_name="06_optional_catboost") as run:
    mlflow.log_params({
        "iterations": 300,
        "depth": 6,
        "learning_rate": 0.05,
        "loss_function": "MAE",
    })
    mlflow.log_metrics(cat_metrics)
    mlflow.set_tags({"stage": "catboost", "optional": "true", "lab": "LR2"})
    if REQUIREMENTS_PATH.exists():
        mlflow.log_artifact(str(REQUIREMENTS_PATH), artifact_path="project")
    mlflow.catboost.log_model(
        cb_model=catboost_model,
        artifact_path="model",
        signature=cat_signature,
        input_example=X_train_cat.head(5),
        registered_model_name=MODEL_NAME,
    )
    run_ids["optional_catboost"] = run.info.run_id

print("CatBoost run_id:", run_ids["optional_catboost"])
print("Зарегистрирована версия модели:", latest_model_version())


/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Registered model 'car_price_prediction_model' already exists. Creating a new version of this model...
2026/09/22 23:50:13 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: car_price_prediction_model, version 3


Created version '3' of model 'car_price_prediction_model'.



2026/09/22 23:50:13 INFO mlflow.tracking._tracking_service.client: 🏃 View run 06_optional_catboost at: http://127.0.0.1:5000/#/experiments/1/runs/e0deea5b776b425dbdcb6f199341b9d9.


2026/09/22 23:50:13 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


CatBoost run_id: e0deea5b776b425dbdcb6f199341b9d9
Зарегистрирована версия модели: 3


## 9. Сравнение всех прогонов


In [18]:
results_df = (
    pd.DataFrame(experiment_results)
      .T
      .sort_values("mae")
)
display(results_df)

results_md = results_df.to_markdown(floatfmt=".6f")
(RESEARCH_DIR / "experiment_results.md").write_text(results_md + "\n", encoding="utf-8")

best_experiment = results_df.index[0]
print("Лучший эксперимент по MAE:", best_experiment)
print(results_df.loc[best_experiment])


,mae,mape,mse
optuna_random_forest,0.632630,0.123129,2.321851
baseline_random_forest,0.636031,0.138466,2.285179
sklearn_feature_engineering,0.659286,0.135843,2.522464
forward_sfs,0.664955,0.131208,2.369676
optional_rfe,0.692535,0.137637,2.393862
optional_catboost,0.779166,0.263721,2.286548


Лучший эксперимент по MAE: optuna_random_forest
mae     0.632630
mape    0.123129
mse     2.321851
Name: optuna_random_forest, dtype: float64


## 10. Production: переобучение лучшей модели на всей выборке

Метрики здесь повторно не измеряются. Логируются модель, сигнатура, пример входных данных, `requirements.txt` и список используемых столбцов. Затем версия модели получает тег и alias `Production`.


In [19]:
PRODUCTION_COLUMNS_PATH = RESEARCH_DIR / "production_columns.txt"
save_lines(PRODUCTION_COLUMNS_PATH, X.columns.tolist())

if best_experiment == "optional_catboost":
    X_full = X.copy()
    for col in categorical_cols:
        X_full[col] = X_full[col].astype(str)

    production_model = CatBoostRegressor(
        iterations=300,
        depth=6,
        learning_rate=0.05,
        loss_function="MAE",
        random_seed=RANDOM_STATE,
        verbose=False,
    )
    production_model.fit(X_full, y, cat_features=categorical_cols)
    input_example = X_full.head(5)
    signature = infer_signature(input_example, production_model.predict(input_example))

    with mlflow.start_run(run_name="07_production_full_data_catboost") as run:
        mlflow.log_params({"source_best_experiment": best_experiment, "train_rows": len(X_full)})
        mlflow.set_tags({"stage": "Production", "lab": "LR2"})
        mlflow.log_artifact(str(REQUIREMENTS_PATH), artifact_path="project")
        mlflow.log_artifact(str(PRODUCTION_COLUMNS_PATH), artifact_path="research")
        mlflow.catboost.log_model(
            cb_model=production_model,
            artifact_path="model",
            signature=signature,
            input_example=input_example,
            registered_model_name=MODEL_NAME,
        )
        production_run_id = run.info.run_id
else:
    # Если лучший результат дал RandomForest, воспроизводим соответствующий вариант.
    if best_experiment == "optuna_random_forest":
        production_model = build_rf_variant(
            best_rf_variant,
            RandomForestRegressor(**best_rf_params)
        )
    elif best_experiment == "baseline_random_forest":
        production_model = build_rf_variant(
            "baseline_random_forest",
            RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
        )
    elif best_experiment == "sklearn_feature_engineering":
        production_model = build_rf_variant(
            "sklearn_feature_engineering",
            RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
        )
    elif best_experiment == "forward_sfs":
        production_model = build_rf_variant(
            "forward_sfs",
            RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
        )
    else:
        production_model = rfe_model

    production_model.fit(X, y)
    input_example = X.head(5)
    signature = infer_signature(input_example, production_model.predict(input_example))

    with mlflow.start_run(run_name="07_production_full_data_random_forest") as run:
        mlflow.log_params({"source_best_experiment": best_experiment, "train_rows": len(X)})
        mlflow.set_tags({"stage": "Production", "lab": "LR2"})
        mlflow.log_artifact(str(REQUIREMENTS_PATH), artifact_path="project")
        mlflow.log_artifact(str(PRODUCTION_COLUMNS_PATH), artifact_path="research")
        mlflow.sklearn.log_model(
            sk_model=production_model,
            artifact_path="model",
            signature=signature,
            input_example=input_example,
            registered_model_name=MODEL_NAME,
        )
        production_run_id = run.info.run_id

production_version = latest_model_version()
client.set_model_version_tag(MODEL_NAME, str(production_version), "Production", "true")
try:
    client.set_registered_model_alias(MODEL_NAME, "Production", str(production_version))
except Exception as exc:
    print("Alias не установлен (тег Production уже установлен):", exc)

(RESEARCH_DIR / "production_run_id.txt").write_text(production_run_id, encoding="utf-8")
print("Production run_id:", production_run_id)
print("Production model version:", production_version)


/home/webber/iis/lab1_car_price/.venv_lab1/lib/python3.10/site-packages/mlflow/types/utils.py:407: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Registered model 'car_price_prediction_model' already exists. Creating a new version of this model...
2026/09/22 23:50:14 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: car_price_prediction_model, version 4


Created version '4' of model 'car_price_prediction_model'.



2026/09/22 23:50:14 INFO mlflow.tracking._tracking_service.client: 🏃 View run 07_production_full_data_random_forest at: http://127.0.0.1:5000/#/experiments/1/runs/267d6c82ea5a442e851f521146fb51c2.


2026/09/22 23:50:14 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


Production run_id: 267d6c82ea5a442e851f521146fb51c2
Production model version: 4


## 11. Сохранение MLmodel для Production-модели


In [20]:
# Сохраняем обязательный MLmodel в research/MLmodel.
import shutil
import tempfile
with tempfile.TemporaryDirectory() as temp_dir:
    downloaded_path = mlflow.artifacts.download_artifacts(
        run_id=production_run_id, artifact_path="model/MLmodel", dst_path=temp_dir,
    )
    shutil.copy2(downloaded_path, RESEARCH_DIR / "MLmodel")
assert (RESEARCH_DIR / "MLmodel").is_file()
print("MLmodel сохранен:", RESEARCH_DIR / "MLmodel")


MLmodel сохранен: /home/webber/iis/lab1_car_price/research/MLmodel


## 12. Автоматическое обновление README результатами ЛР2


In [21]:
# Формируем блок с фактическими результатами текущего запуска.
best_row = results_df.loc[best_experiment]
selected_for_readme = selected_names if best_experiment == "forward_sfs" else X.columns.tolist()

if best_experiment == "optuna_random_forest":
    params_text = json.dumps(study.best_params, ensure_ascii=False)
elif best_experiment == "optional_catboost":
    params_text = json.dumps({"iterations": 300, "depth": 6, "learning_rate": 0.05}, ensure_ascii=False)
else:
    params_text = "см. research/research.ipynb"

block = f"""<!-- LR2_AUTO_START -->
### Результаты исследования

Лучший результат на тестовой выборке показал **{best_experiment}**.

- MAE: **{best_row['mae']:.6f}**
- MAPE: **{best_row['mape']:.6f}**
- MSE: **{best_row['mse']:.6f}**
- параметры: `{params_text}`
- Production model version: **{production_version}**
- `run_id = {production_run_id}`

Используемые исходные столбцы: `{', '.join(X.columns)}`.
<!-- LR2_AUTO_END -->"""

text = README_PATH.read_text(encoding="utf-8")
if "<!-- LR2_AUTO_START -->" in text:
    text = re.sub(r"<!-- LR2_AUTO_START -->.*?<!-- LR2_AUTO_END -->", lambda match: block, text, flags=re.S)
else:
    text += "\n\n" + block + "\n"
README_PATH.write_text(text, encoding="utf-8")
print("README.md обновлен фактическими результатами и Production run_id.")


README.md обновлен фактическими результатами и Production run_id.


## 13. Финальные действия

1. В MLflow открыть эксперимент и сохранить `research/model_runs.png` со всеми run и метриками/графиками.
2. Открыть Model Registry и сохранить `research/model_versions.png` с версиями зарегистрированной модели.
3. Проверить наличие `research/MLmodel`.
4. Проверить, что `mlflow/start_mlflow.sh` находится в Git, а `mlflow/mlruns.db` и `mlflow/mlartifacts/` игнорируются.
5. Сделать финальный коммит `LR2_final` и отправить изменения на GitHub.


# Контрольные вопросы — краткие ответы

**1. Для чего нужен feature extraction?**  
Для построения новых информативных признаков из исходных данных, чтобы модель могла лучше описывать зависимости.

**2. Для чего нужен feature selection?**  
Для отбора наиболее полезных признаков, уменьшения размерности, времени обучения и риска переобучения.

**3. Для чего используется MLflow?**  
Для учета экспериментов: параметров, метрик, артефактов, моделей и их версий, а также для воспроизводимости и Model Registry.

**4. Что такое гиперпараметры модели и как их настраивать?**  
Это параметры алгоритма, которые задаются до обучения. Их выбирают сравнением качества на validation/CV; в данной работе используется Optuna и минимизация MAE.

**5. Как выглядит pipeline модели?**  
Обычно: входные данные → preprocessing/encoding/scaling → feature engineering → feature selection → модель → prediction. Не все шаги обязательны для каждой задачи.
